## Installations

In [1]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1" "onnxruntime-gpu==1.20.1"
!pip install google-cloud-texttospeech

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 kB 10.7 MB/s eta 0:00:00


## Login and API keys

**Important**
- You should log in via your google cloud console account
- In your cloud console account you have to configure and project with Google Cloud Text to Speech Enabled
- You should put that project id in the command below


- Also, you need hf token and open_ai_api key
- You have to accept terms and conditions for the model (https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual)


**Required files**

Four files should be in the current working directory
1. `output_union_damage.csv`
2. `conversation script.txt`
3. `Cyclone guideline instruction.txt`
4. `flood guideline instruction.txt`

(All of these files are in the github repository)


In [2]:
!gcloud auth login
!gcloud config set project mapathon-491315
!gcloud auth application-default login
!gcloud auth application-default set-quota-project mapathon-491315

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=l573sovkoFw68uvQROv7LZdjI61W35&prompt=consent&token_usage=remote&access_type=offline&code_challenge=O0JKwzT5X2MtNB84AEVJxyn8uMySk8I09Md706y9alU&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E8mKq-t_p3Wdny7-sFsx_TgHV8XmdAZMknh8jTHDMvIKmuQvdIfOsRP1xBbaZsVzA

You are now logged in as [saintworrior249@gmail.com].
Your current proj

In [3]:
import os
os.environ['OPENAI_API_KEY'] = "" ## Your openai api key here
os.environ['HF_TOKEN'] = "" ## Your huggingface token here

In [4]:
from transformers import AutoModel
import torch, torchaudio

# Load the model
model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True,
    token = os.environ['HF_TOKEN'],
    device_map = 'auto'
)

config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:115: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [13]:
from IPython.display import Javascript, display, Audio
from google.colab import output
from base64 import b64decode
import scipy.io.wavfile as wavfile
import io
import numpy as np

RECORD_JS = r"""
async function recordAudio() {
  const div = document.createElement('div');
  const startBtn = document.createElement('button');
  const stopBtn = document.createElement('button');
  const endBtn = document.createElement('button');
  const status = document.createElement('span');

  startBtn.textContent = 'Start recording';
  stopBtn.textContent = 'Stop';
  endBtn.textContent = 'End Call';

  stopBtn.disabled = true;

  status.textContent = ' Idle';

  div.appendChild(startBtn);
  div.appendChild(stopBtn);
  div.appendChild(endBtn);
  div.appendChild(status);
  document.body.appendChild(div);

  let stream;
  let recorder;
  let chunks = [];

  const audioData = await new Promise((resolve) => {

    startBtn.onclick = async () => {
      stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      recorder = new MediaRecorder(stream);
      chunks = [];

      recorder.ondataavailable = e => {
        if (e.data.size > 0) chunks.push(e.data);
      };

      recorder.onstop = async () => {
        const blob = new Blob(chunks, { type: 'audio/webm' });
        const reader = new FileReader();
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);

        if (stream) {
          stream.getTracks().forEach(track => track.stop());
        }
        div.remove();
      };

      recorder.start();
      startBtn.disabled = true;
      stopBtn.disabled = false;
      status.textContent = ' Recording...';
    };

    stopBtn.onclick = () => {
      stopBtn.disabled = true;
      status.textContent = ' Processing...';
      recorder.stop();
    };

    // 🔴 End Call button
    endBtn.onclick = () => {
      if (recorder && recorder.state !== "inactive") {
        recorder.stop();
      }
      if (stream) {
        stream.getTracks().forEach(track => track.stop());
      }
      div.remove();
      resolve(null);  // return None to Python
    };

  });

  return audioData;
}
"""

def record_audio(filename="recorded.wav"):
    display(Javascript(RECORD_JS))
    data = output.eval_js("recordAudio()")

    # 🔴 Handle End Call
    if data is None:
        return None

    binary = b64decode(data.split(",")[1])

    with open(filename, "wb") as f:
        f.write(binary)

    return filename

In [33]:
# Load an audio file
import time

def speech_recognize(file_name):
  wav, sr = torchaudio.load(file_name)
  wav = torch.mean(wav, dim=0, keepdim=True)

  target_sample_rate = 16000  # Expected sample rate
  if sr != target_sample_rate:
      resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
      wav = resampler(wav)

  # # Perform ASR with CTC decoding
  # transcription_ctc = model(wav, "bn", "ctc",)
  # print("CTC Transcription:", transcription_ctc)
  # print(f"ctc required: {time.time() - start_time} sec")

  # Perform ASR with RNNT decoding
  transcription_rnnt = model(wav, "bn", "rnnt")
  return transcription_rnnt

In [15]:
from openai import OpenAI

client = OpenAI()

In [16]:
refiner_system_prompt = (
    "You are an AI text input refining and interpretation assistant.\n"
    "You will given input and text in bengali language, which is output of an Automatic Speech Recognition (ASR) model. "
    "The ASR doesn't put any puncutaion marks, it may make slight mistake and writing may be in any local accent of Bangladesh."
    "You may sometimes be provided with previous conversation for context understanding."
    "Your task is to understand the input text, refine that and write back that with in good bengali language with proper punctuation mark."

    "Only write back the refined text as output."
)


In [17]:
def refine_user_input(text, context = None):
  if context:
    user_message = f"##previous converstaion: \n\n {context}\n"
  else:
    user_message = ""
  user_message += f"##Current ASR output to refine: \n\n {text}"

  response = client.responses.create(
      model="gpt-5.4-mini",
      input=[
          {'role': 'system', 'content': refiner_system_prompt},
          {'role': 'user', 'content': user_message}
      ]
  )

  refined_input = response.output_text
  return refined_input

In [18]:
with open("Cyclone guideline instruction.txt", 'r') as file:
  guideline = file.read()

with open("conversation script.txt", 'r') as file:
  sample_conversation = file.read()


storm_information = ""

In [19]:
import pandas as pd

damage_output = pd.read_csv("output_union_damage.csv")

target_union = "Amtali"
target_upazilla = 'Amtali'

selected_row = damage_output[
    (damage_output['Union'] == target_union) &
    (damage_output['Upazilla'] == target_upazilla)
].iloc[0]

bmd_signal_no = 8
lead_time_hours = 36


storm_information = (
    f"storm type: Cyclone\n"
    f"Storm speed: {selected_row['Storm Speed (m/s)']} m/s\n"
    f"Rainfall: {selected_row['Rainfall (mm)']} mm\n"
    f"Storm Surge: {selected_row['Storm Surge (m)']} m\n"
    f"Lead time: {lead_time_hours} hours\n"
    f"BMD signal No: {bmd_signal_no}\n"
)

assistant_identity = f"You are an conversational AI assistant to spread out awareness about upcoming cyclone. A cyclone is about to happen within {lead_time_hours} hours.\n"

must_follow_addons = ""
if bmd_signal_no >= 8:
  must_follow_addons += "- Ask the person to immediately leave the place.\n"

In [20]:
print(storm_information)

storm type: Cyclone
Storm speed: 17.1756 m/s
Rainfall: 61.6862 mm
Storm Surge: 0.1987 m
Lead time: 36 hours
BMD signal No: 8



In [21]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class AgentResponse(BaseModel):
    plain_text: str
    ssml_text: str
    call_end: Optional[bool]

In [22]:
assistant_system_prompt = (
    f"{assistant_identity}"
    "The response you give will be converted to voice using text-to-speech, so do not use emoji in you respone and keep your response voice friendly.\n"
    "You will ultimately spread awareness and give guidance to the user using this guideline:\n"
    f"{guideline}\n"
    "A sample conversation is given below. Follow the components of this conversation:\n"
    f"{sample_conversation}\n"
    "By the conversation you will find out these informations."

    "Storm information in the user's location:\n"
    f"{storm_information}"

    "Always be respectful to the user and follow bangladeshi manners.\n"
    "Do not directly call the user by his/her name. Instead use one of the 'স্যার/ভাই/আপু/ম্যাডাম' when necessary depending on context and gender."
    "Always use respect full terms like 'আপনি'. "
    "Keep your tone convincing, so that user do not panic."

    "Once guideline giving and all information collection is completed naturally say goodbye and complete the conversation."

    # "Output format:\n"
    # "- Generate SSML text of your to be used in google cloud text to speech.\n"
    # "- Put proper intonation, stops, ups downs of pitch in the SSML to make the voice sound natural."
    # "- Response in specified structured format.\n"
    # "- Trigger end call true in the structure when call is ended.\n\n"

    'Must follow these:\n'
    '- Do not ask multiple questions at once.\n'
    '- Do not bullet point items. Keep response paragraph style.\n'
    '- Keep your response as concise as possible at the begining and during questionaries.\n'
    '- Do not give elaborated guideline after the each questions. Ask relavent follow up question. Like, if the user says he is fisherman, ask where he currently is, whether he is in sea currently? and give guideline by these follow up style.\n'
    '- Give elaborated guideline once you have completed gathering all of the informations.\n'
    '- Keep the conversation lively.\n'
    "- during conversation ending use words like 'সাবধানে থাকুন', 'সতর্ক থাকবেন', 'আসসালামু আলাইকুম'"
    f"{must_follow_addons}"
    )

In [35]:
def generate_agent_response(input):
  if isinstance(input, str):
    input = [{'role': 'user', 'content': input}]

  response = client.responses.create(
      model="gpt-5.4-mini",
      input=[
          {'role': 'system', 'content': assistant_system_prompt}
      ] + input,
      temperature = 0.7,
      # frequency_penalty=0.8,
      # presence_penalty=0.3,
  )

  agent_reply = response.output_text
  return agent_reply

## Google TTS

In [37]:
from google.cloud import texttospeech
from IPython.display import Audio

tts_client = texttospeech.TextToSpeechClient()

voice = texttospeech.VoiceSelectionParams(
    language_code="bn-BD",
    # name = "bn-IN-Chirp3-HD-Gacrux",
    ssml_gender=texttospeech.SsmlVoiceGender.FEMALE
)

audio_config = texttospeech.AudioConfig(
    audio_encoding=texttospeech.AudioEncoding.MP3,
    speaking_rate = 1.2,
    volume_gain_db=2.0
)

def text_to_speech(agent_reply, file_name = "output.mp3"):
  input_text = texttospeech.SynthesisInput(text=agent_reply)

  response = tts_client.synthesize_speech(
      input=input_text, voice=voice, audio_config=audio_config
  )

  with open(file_name, "wb") as out:
      out.write(response.audio_content)

  display(Audio(file_name, autoplay = True))
  return file_name

## Final pipeline

In [29]:
info_table = """
| Field | Value (choose or specify) |
|------|--------------------------|
| **call_status** | completed / partial / failed / callback |
| **warning_awareness** | yes / no / unsure |
| **location** | district, upazila, union, village (optional GPS) |
| **housing_type** | jhupri / kacha / semi-pucca / pucca |
| **water_source_risk** | tube well protected / unprotected / already contaminated / unknown |
| **latrine_risk** | stable / flood-prone / damaged / unknown |
| **livelihood** | fisherman / fish farmer / farmer / salt farmer / livestock owner / other |
| **vulnerable_members** | elderly / disability / pregnancy / children / chronic illness |
| **can_evacuate** | yes / no / with assistance |
| **current_local_condition** | no impact / water rising / strong wind / road blocked / embankment issue |
| **asset_at_risk** | livestock / nets / boat / seed / crop / documents / other |
| **priority_class** | low / medium / high / urgent |
| **recommended_followup** | human callback / volunteer referral / shelter info / medical support / none |
"""

extractor_system_prompt = (
    "Your job is to extract some info in specified structure from an conversation.\n"
    "The info you will extract are given below with specified details:\n"
    f"{info_table}"
)

In [2]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class Location(BaseModel):
    district: str
    upazila: str
    union: str
    village: str
    gps: Optional[str] = Field(
        default=None,
        description="Optional GPS coordinates or map reference"
    )


class StructuredInput(BaseModel):
    call_status: Literal["completed", "partial", "failed", "callback"]
    warning_awareness: Literal["yes", "no", "unsure"]
    location: Location
    housing_type: Literal["jhupri", "kacha", "semi-pucca", "pucca"]
    water_source_risk: Literal[
        "tube well protected",
        "unprotected",
        "already contaminated",
        "unknown",
    ]
    latrine_risk: Literal["stable", "flood-prone", "damaged", "unknown"]
    livelihood: Literal[
        "fisherman",
        "fish farmer",
        "farmer",
        "salt farmer",
        "livestock owner",
        "other",
    ]
    vulnerable_members: List[
        Literal["elderly", "disability", "pregnancy", "children", "chronic illness"]
    ] = Field(default_factory=list)
    can_evacuate: Literal["yes", "no", "with assistance"]
    current_local_condition: Literal[
        "no impact",
        "water rising",
        "strong wind",
        "road blocked",
        "embankment issue",
    ]
    asset_at_risk: List[
        Literal["livestock", "nets", "boat", "seed", "crop", "documents", "other"]
    ] = Field(default_factory=list)
    priority_class: Literal["low", "medium", "high", "urgent"]
    recommended_followup: Literal[
        "human callback",
        "volunteer referral",
        "shelter info",
        "medical support",
        "none",
    ]

In [ ]:
import time
import os

running_messages = []
turn = 0

os.makedirs("voice_files", exist_ok = True)

print("+++ Welcome to voice AI agent demonstation +++")
print("To use this, you have press the 'start' button to record you're audo and and 'stop' to end recording")
print("Once you have finished recording an audio, it will transcribe it and send it to LLM.")
print("After that LLM will generate a response for you and that will be automatically played")

print("Once that LLM reply is complete you can record and send your next message.")

while True:
  turn += 1
  # Audio input
  webm_file = record_audio(f"voice_files/user_voice_turn_{turn}.mp3")
  if not webm_file:
    break

  #ASR
  user_text = speech_recognize(webm_file)
  refined_text = refine_user_input(user_text)
  print(f"User : {refined_text}")

  running_messages.append(
      {'role' : 'user', 'content' : refined_text}
  )

  ai_reply = generate_agent_response(running_messages)
  print(f"Agent: {ai_reply}")

  running_messages.append(
      {'role' : 'assistant', 'content' : ai_reply}
  )

  output_file_name = text_to_speech(ai_reply, f"voice_files/agent_voice_turn_{turn}.mp3")

  print('\n\n')
  time.sleep(1)

In [40]:
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()

response = client.responses.parse(
    model="gpt-5.4",
    input=[
        {"role": "system", "content": extractor_system_prompt},
        {
            "role": "user",
            "content": str(running_messages)
        },
    ],
    text_format=StructuredInput,
)

event = response.output_parsed

print("Structured output")
print()
print()
display(dict(event))

{'call_status': 'completed',
 'warning_awareness': 'yes',
 'location': Location(district='', upazila='', union='', village='', gps=None),
 'housing_type': 'semi-pucca',
 'water_source_risk': 'unprotected',
 'latrine_risk': 'flood-prone',
 'livelihood': 'fisherman',
 'vulnerable_members': ['elderly'],
 'can_evacuate': 'no',
 'current_local_condition': 'no impact',
 'asset_at_risk': ['boat', 'nets', 'other'],
 'priority_class': 'high',
 'recommended_followup': 'shelter info'}

In [44]:
with open("conversation_log.txt", 'w') as file:
  for msg in running_messages:
    print(
        f"{msg['role']}: {msg['content']}\n\n",
        file = file
    )
  print("\n\n\n\n\n", file = file)
  print("=== Extracted Data ===\n", file = file)
  print(dict(event), file = file)